# BOXER Format Round-Trip Tests

Tests for the BOXER card-image (ASCII) covariance format read/write.

- **Test 1** — Compression/decompression unit tests (identity, random, sparse)
- **Test 2** — BOXER round-trip: `read_covfil` → `write_boxer` → `read_boxer` → compare
- **Test 3** — Cross-format round-trip: covfil → boxer → covfil → compare
- **Test 4** — Inspect written BOXER file (visual check)
- **Test 5** — Symmetric vs non-symmetric detection
- **Test 6** — CovMat class method API

In [ ]:
import tempfile
import os
import numpy as np

from kika.cov import read_covfil, write_covfil, read_boxer, write_boxer
from kika.cov.covmat import CovMat
from kika.cov.parse_covmat import (
    _compress_boxer, _decompress_boxer,
    _format_boxer_header, _parse_boxer_header,
    _choose_boxer_ncf,
)

MF33_FILE = r'E:\OneDrive\Share\Juan\260560_40.02.xs.gendf'

## Test 1 — Compression / Decompression Unit Tests

In [ ]:
# --- 1a: Identity matrix (symmetric) ---
n = 10
eye = np.eye(n)
xval, icons = _compress_boxer(eye, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(eye, result), f'Identity mismatch: max_diff={np.max(np.abs(eye - result)):.2e}'
print(f'1a Identity ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1b: Random symmetric matrix ---
np.random.seed(42)
n = 20
A = np.random.randn(n, n)
A = (A + A.T) / 2  # symmetrize
xval, icons = _compress_boxer(A, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(A, result), f'Random symmetric mismatch: max_diff={np.max(np.abs(A - result)):.2e}'
print(f'1b Random symmetric ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1c: Sparse symmetric matrix ---
n = 15
S = np.zeros((n, n))
S[3, 3] = 1.5
S[3, 5] = 0.3; S[5, 3] = 0.3
S[5, 5] = 2.0
S[10, 10] = 0.7
xval, icons = _compress_boxer(S, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(S, result), f'Sparse mismatch: max_diff={np.max(np.abs(S - result)):.2e}'
print(f'1c Sparse symmetric ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1d: Non-symmetric matrix ---
n = 8
B = np.random.randn(n, n)
xval, icons = _compress_boxer(B, symmetric=False)
result = _decompress_boxer(xval, icons, n, n)
assert np.allclose(B, result), f'Non-symmetric mismatch: max_diff={np.max(np.abs(B - result)):.2e}'
print(f'1d Non-symmetric ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1e: Zero matrix ---
n = 5
Z = np.zeros((n, n))
xval, icons = _compress_boxer(Z, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(Z, result), 'Zero matrix mismatch'
print(f'1e Zero matrix ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1f: Matrix with carry-down pattern ---
n = 6
C = np.ones((n, n)) * 3.14
xval, icons = _compress_boxer(C, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(C, result), 'Carry-down mismatch'
print(f'1f Carry-down ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

print()
print('All compression/decompression unit tests passed.')

## Test 2 — BOXER Round-Trip (covfil → boxer → compare)

In [ ]:
# Read MF33 COVFIL
mf33 = read_covfil(MF33_FILE)
print(f'Original: {mf33.num_matrices} matrices, {mf33.num_groups} groups, '
      f'{len(mf33.cross_sections)} XS entries')

# Write to BOXER
tmp_boxer = tempfile.mktemp(suffix='.boxer')
write_boxer(mf33, tmp_boxer, hlibid='TST', hdescr='round-trip test', nvf=10)

# Read back from BOXER
mf33_rt = read_boxer(tmp_boxer)
print(f'Round-trip: {mf33_rt.num_matrices} matrices, {mf33_rt.num_groups} groups, '
      f'{len(mf33_rt.cross_sections)} XS entries')

# Compare metadata
assert mf33_rt.num_groups == mf33.num_groups, \
    f'Groups mismatch: {mf33_rt.num_groups} vs {mf33.num_groups}'
assert mf33_rt.num_matrices == mf33.num_matrices, \
    f'Matrix count mismatch: {mf33_rt.num_matrices} vs {mf33.num_matrices}'

# Compare energy grid (BOXER NVF=10 gives ~4 sig figs)
assert np.allclose(mf33.energy_grid, mf33_rt.energy_grid, rtol=1e-3), \
    'Energy grid mismatch'
print('Energy grid: MATCH (rtol=1e-3)')

# Compare matrices
for i in range(mf33.num_matrices):
    mt_r = mf33.reaction_rows[i]
    mt_c = mf33.reaction_cols[i]
    orig = mf33.matrices[i]
    rt = mf33_rt.matrices[i]
    
    # Use relative tolerance where values are significant
    mask = np.abs(orig) > 1e-30
    if mask.any():
        rel_diff = np.max(np.abs((orig[mask] - rt[mask]) / orig[mask]))
    else:
        rel_diff = 0.0
    abs_diff = np.max(np.abs(orig - rt))
    
    print(f'  MT=({mt_r},{mt_c}): max_rel={rel_diff:.2e}, max_abs={abs_diff:.2e}')
    assert rel_diff < 1e-2 or abs_diff < 1e-6, \
        f'Matrix {i} MT=({mt_r},{mt_c}) mismatch: rel={rel_diff:.2e} abs={abs_diff:.2e}'

# Compare cross sections
for key in mf33.cross_sections:
    assert key in mf33_rt.cross_sections, f'Missing XS key: {key}'
    orig_xs = mf33.cross_sections[key]
    rt_xs = mf33_rt.cross_sections[key]
    assert np.allclose(orig_xs, rt_xs, rtol=1e-3), \
        f'XS mismatch for {key}: max_diff={np.max(np.abs(orig_xs - rt_xs)):.2e}'
print(f'Cross sections ({len(mf33.cross_sections)} entries): MATCH (rtol=1e-3)')

os.unlink(tmp_boxer)
print()
print('BOXER round-trip test PASSED.')

## Test 3 — Cross-Format Round-Trip (covfil → boxer → covfil)

In [ ]:
# covfil -> boxer -> covfil -> compare
mf33 = read_covfil(MF33_FILE)

# Step 1: write boxer
tmp_boxer = tempfile.mktemp(suffix='.boxer')
write_boxer(mf33, tmp_boxer, nvf=10)

# Step 2: read boxer
mf33_boxer = read_boxer(tmp_boxer)

# Step 3: write covfil from boxer data
tmp_covfil = tempfile.mktemp(suffix='.gendf')
write_covfil(mf33_boxer, tmp_covfil, tape_label='cross-format test')

# Step 4: read covfil back
mf33_final = read_covfil(tmp_covfil)

# Compare original vs final
assert mf33_final.num_matrices == mf33.num_matrices
assert mf33_final.num_groups == mf33.num_groups

for i in range(mf33.num_matrices):
    orig = mf33.matrices[i]
    final = mf33_final.matrices[i]
    mask = np.abs(orig) > 1e-30
    if mask.any():
        rel_diff = np.max(np.abs((orig[mask] - final[mask]) / orig[mask]))
    else:
        rel_diff = 0.0
    mt_r = mf33.reaction_rows[i]
    mt_c = mf33.reaction_cols[i]
    print(f'  MT=({mt_r},{mt_c}): max_rel_diff={rel_diff:.2e}')
    assert rel_diff < 1e-2 or np.max(np.abs(orig - final)) < 1e-6

os.unlink(tmp_boxer)
os.unlink(tmp_covfil)
print()
print('Cross-format round-trip test PASSED.')

## Test 4 — Inspect Written BOXER File

In [ ]:
# Write boxer and inspect first 30 lines
mf33 = read_covfil(MF33_FILE)
tmp_boxer = tempfile.mktemp(suffix='.boxer')
write_boxer(mf33, tmp_boxer, hlibid='K56', hdescr='Fe-56 covariance test', nvf=10)

with open(tmp_boxer, 'r') as f:
    lines = f.readlines()

print(f'Total lines: {len(lines)}')
print(f'File size: {os.path.getsize(tmp_boxer)} bytes')
print()
print('First 30 lines:')
print('-' * 80)
for i, line in enumerate(lines[:30]):
    print(f'{i+1:4d} | {line.rstrip()}')
print('-' * 80)

# Parse and display all headers
print()
print('Block headers:')
cursor = 0
while cursor < len(lines):
    hdr = _parse_boxer_header(lines[cursor])
    itype = hdr['itype']
    itype_names = {0: 'ENERGY', 1: 'XS', 2: 'STD-DEV', 3: 'COV', 4: 'CORR'}
    print(f'  Line {cursor+1:4d}: ITYPE={itype} ({itype_names.get(itype, "?"):8s}) '
          f'MAT={hdr["mat"]:5d} MT={hdr["mt"]:3d} MAT1={hdr["mat1"]:5d} MT1={hdr["mt1"]:3d} '
          f'NVAL={hdr["nval"]:4d} NVF={hdr["nvf"]:2d} NCON={hdr["ncon"]:4d} NCF={hdr["ncf"]:1d} '
          f'NROWM={hdr["nrowm"]:3d} NROWH={hdr["nrowh"]:3d} NCOLH={hdr["ncolh"]:3d}')
    cursor += 1
    # Skip value and control lines
    from kika.cov.parse_covmat import _BOXER_VALUE_FORMATS, _BOXER_CONTROL_FORMATS
    import math
    if hdr['nval'] > 0:
        vpl, _ = _BOXER_VALUE_FORMATS[hdr['nvf']]
        cursor += math.ceil(hdr['nval'] / vpl)
    if hdr['ncon'] > 0:
        vpl, _ = _BOXER_CONTROL_FORMATS[hdr['ncf']]
        cursor += math.ceil(hdr['ncon'] / vpl)

os.unlink(tmp_boxer)

## Test 5 — Symmetric vs Non-Symmetric Detection

In [ ]:
mf33 = read_covfil(MF33_FILE)
tmp_boxer = tempfile.mktemp(suffix='.boxer')
write_boxer(mf33, tmp_boxer, nvf=10)

with open(tmp_boxer, 'r') as f:
    lines = f.readlines()

# Check each ITYPE=3 header for correct NCOLH
import math
from kika.cov.parse_covmat import _BOXER_VALUE_FORMATS, _BOXER_CONTROL_FORMATS

cursor = 0
while cursor < len(lines):
    hdr = _parse_boxer_header(lines[cursor])
    cursor += 1
    if hdr['nval'] > 0:
        vpl, _ = _BOXER_VALUE_FORMATS[hdr['nvf']]
        cursor += math.ceil(hdr['nval'] / vpl)
    if hdr['ncon'] > 0:
        vpl, _ = _BOXER_CONTROL_FORMATS[hdr['ncf']]
        cursor += math.ceil(hdr['ncon'] / vpl)
    
    if hdr['itype'] == 3:
        is_diag = (hdr['mat'] == hdr['mat1'] and hdr['mt'] == hdr['mt1'])
        if is_diag:
            assert hdr['ncolh'] == 0, \
                f'Diagonal block MAT={hdr["mat"]} MT={hdr["mt"]} should have NCOLH=0, got {hdr["ncolh"]}'
            print(f'  Diagonal: MAT={hdr["mat"]} MT={hdr["mt"]} -> NCOLH=0 (symmetric) OK')
        else:
            assert hdr['ncolh'] > 0, \
                f'Off-diagonal block should have NCOLH>0, got {hdr["ncolh"]}'
            print(f'  Off-diag: MAT={hdr["mat"]} MT={hdr["mt"]} x MAT1={hdr["mat1"]} MT1={hdr["mt1"]} -> NCOLH={hdr["ncolh"]} OK')

os.unlink(tmp_boxer)
print()
print('Symmetric vs non-symmetric detection test PASSED.')

## Test 6 — CovMat Class Method API

In [ ]:
# Load via covfil class method
mf33 = CovMat.from_covfil(MF33_FILE)

# Write via .to_boxer()
tmp_boxer = tempfile.mktemp(suffix='.boxer')
mf33.to_boxer(tmp_boxer, hlibid='API', nvf=10)

# Read via CovMat.from_boxer()
mf33_rt = CovMat.from_boxer(tmp_boxer)

assert isinstance(mf33_rt, CovMat)
assert mf33_rt.num_groups == mf33.num_groups
assert mf33_rt.num_matrices == mf33.num_matrices
assert np.allclose(mf33.energy_grid, mf33_rt.energy_grid, rtol=1e-3)

for i in range(mf33.num_matrices):
    orig = mf33.matrices[i]
    rt = mf33_rt.matrices[i]
    mask = np.abs(orig) > 1e-30
    if mask.any():
        rel_diff = np.max(np.abs((orig[mask] - rt[mask]) / orig[mask]))
    else:
        rel_diff = 0.0
    assert rel_diff < 1e-2, f'Matrix {i} class method mismatch: rel={rel_diff:.2e}'

os.unlink(tmp_boxer)
print(f'CovMat.from_boxer() / .to_boxer(): {mf33_rt.num_matrices} matrices, all match.')
print()
print('All CovMat class method API tests PASSED.')

## Test 7 — Header Parse/Format Round-Trip

In [ ]:
# Verify header formatting and parsing are inverse operations
hdr_args = dict(
    itype=3, mat=2631, mt=102, mat1=2631, mt1=102,
    nval=156, nvf=10, ncon=42, ncf=4,
    nrowm=0, nrowh=56, ncolh=0,
    hlib='K56', hdescr='Fe-56 test library',
)
line = _format_boxer_header(**hdr_args)
assert len(line) == 80, f'Header length is {len(line)}, expected 80'

parsed = _parse_boxer_header(line)
assert parsed['itype'] == hdr_args['itype']
assert parsed['mat'] == hdr_args['mat']
assert parsed['mt'] == hdr_args['mt']
assert parsed['mat1'] == hdr_args['mat1']
assert parsed['mt1'] == hdr_args['mt1']
assert parsed['nval'] == hdr_args['nval']
assert parsed['nvf'] == hdr_args['nvf']
assert parsed['ncon'] == hdr_args['ncon']
assert parsed['ncf'] == hdr_args['ncf']
assert parsed['nrowm'] == hdr_args['nrowm']
assert parsed['nrowh'] == hdr_args['nrowh']
assert parsed['ncolh'] == hdr_args['ncolh']
assert parsed['hlib'].strip() == 'K56'

print(f'Header: "{line}"')
print(f'Parsed back: {parsed}')
print()
print('Header parse/format round-trip PASSED.')